In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)

    if depth > 4:
        dirs[:] = []
        continue

    print("  " * depth, os.path.basename(root) or "input")

    for name in files[:5]:
        print("  " * (depth + 1), name)

In [ ]:
import json
from pathlib import Path
from collections import Counter

BASE = Path(
    "/kaggle/input/competitions/"
    "filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026"
)

TRAIN_IMAGES = BASE / "train/train_images"
TEST_IMAGES = BASE / "test/test_images"

JSON_PATH = (
    BASE / "train/"
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

with open(JSON_PATH, "r") as f:
    data = json.load(f)

print("Training image files:", len(list(TRAIN_IMAGES.glob("*.jpeg"))))
print("Test image files:", len(list(TEST_IMAGES.glob("*.jpeg"))))

print("Annotation image records:", len(data["images"]))
print("Annotated filaments:", len(data["annotations"]))

print("\nCategories:")
for category in data["categories"]:
    print(category["id"], category["name"])

print("\nFirst image record:")
print(data["images"][0])

print("\nFirst annotation fields:")
print(list(data["annotations"][0].keys()))

print("\nFilaments per annotation-image record:")
counts = Counter(a["image_id"] for a in data["annotations"])
print("Minimum:", min(counts.values()))
print("Maximum:", max(counts.values()))
print("Average:", round(sum(counts.values()) / len(counts), 2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict
import numpy as np

# Group annotations by annotation-image ID
annotations_by_image = defaultdict(list)

for ann in data["annotations"]:
    annotations_by_image[ann["image_id"]].append(ann)

# Select an annotated record containing multiple filaments
image_record = next(
    img for img in data["images"]
    if len(annotations_by_image[img["id"]]) >= 5
    and (TRAIN_IMAGES / img["file_name"]).exists()
)

image_path = TRAIN_IMAGES / image_record["file_name"]

# Grayscale (تدرج رمادي) — one intensity channel
image = np.array(Image.open(image_path).convert("L"))

annotations = annotations_by_image[image_record["id"]]

category_names = {
    c["id"]: c["name"] for c in data["categories"]
}

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original H-alpha Solar Image")

axes[1].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Expert Filament Annotations")

for ann in annotations:
    # Polygon: x0,y0,x1,y1,...
    segments = ann["segmentation"]

    for segment in segments:
        coords = np.asarray(segment).reshape(-1, 2)

        polygon = patches.Polygon(
            coords,
            closed=True,
            facecolor="cyan",
            edgecolor="yellow",
            alpha=0.45,
            linewidth=1
        )

        axes[1].add_patch(polygon)

    x, y, w, h = ann["bbox"]

    box = patches.Rectangle(
        (x, y),
        w, h,
        fill=False,
        edgecolor="orange",
        linewidth=1
    )

    axes[1].add_patch(box)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image:", image_record["file_name"])
print("Image shape:", image.shape)
print("Annotation record:", image_record["id"])
print("Number of annotated filaments:", len(annotations))

for ann in annotations[:5]:
    print(
        category_names[ann["category_id"]],
        "area:", round(ann["area"], 1)
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

records = []

for ann in data["annotations"]:
    x, y, w, h = ann["bbox"]
    area = ann["area"]

    records.append({
        "width": w,
        "height": h,
        "mask_area": area,
        "box_area": w * h,
        "fill_ratio": area / (w * h) if w * h > 0 else np.nan,
    })

df = pd.DataFrame(records)

print("Total filament annotations:", len(df))

print(
    df[["width", "height", "mask_area", "fill_ratio"]]
    .describe(percentiles=[0.1, 0.5, 0.9])
    .round(2)
)

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df["fill_ratio"].dropna(), bins=40)

ax.set_title("How Much of Each Bounding Box Is Actually Filament?")
ax.set_xlabel("Mask area / Bounding box area")
ax.set_ylabel("Number of annotated filaments")

plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

# Unique physical solar image filenames
filenames = sorted({
    img["file_name"] for img in data["images"]
})

train_files, val_files = train_test_split(
    filenames,
    test_size=0.20,
    random_state=42
)

train_files = set(train_files)
val_files = set(val_files)

# Check for overlap
assert train_files.isdisjoint(val_files)

# Assign annotation records to the appropriate split
train_records = [
    img for img in data["images"]
    if img["file_name"] in train_files
]

val_records = [
    img for img in data["images"]
    if img["file_name"] in val_files
]

print("Unique physical images:", len(filenames))
print("Training images:", len(train_files))
print("Validation images:", len(val_files))
print("Training annotation records:", len(train_records))
print("Validation annotation records:", len(val_records))
print("Image overlap:", len(train_files & val_files))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Use the previously selected solar image and its annotations
binary_mask = Image.new("L", (2048, 2048), 0)
draw = ImageDraw.Draw(binary_mask)

for ann in annotations:
    for polygon in ann["segmentation"]:
        points = np.asarray(polygon).reshape(-1, 2)
        draw.polygon(
            [tuple(map(float, p)) for p in points],
            fill=1
        )

mask = np.asarray(binary_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original Image")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Ground-Truth Binary Mask")

axes[2].imshow(image, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(mask == 0, mask),
    cmap="autumn",
    alpha=0.6
)
axes[2].set_title("Image + Mask Overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Mask shape:", mask.shape)
print("Unique values:", np.unique(mask))
print("Filament pixels:", np.count_nonzero(mask))
print("Background pixels:", np.count_nonzero(mask == 0))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PATCH_SIZE = 512

# Find the first filament pixel
ys, xs = np.where(mask == 1)

# Choose a filament pixel near the centre of the detected region
i = len(xs) // 2
cx, cy = int(xs[i]), int(ys[i])

# Calculate top-left corner of the crop
x0 = np.clip(cx - PATCH_SIZE // 2, 0, image.shape[1] - PATCH_SIZE)
y0 = np.clip(cy - PATCH_SIZE // 2, 0, image.shape[0] - PATCH_SIZE)

# Crop exactly the same coordinates from image and mask
image_patch = image[y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE]
mask_patch = mask[y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(image_patch, cmap="gray")
axes[0].set_title("Input: Solar Patch")

axes[1].imshow(mask_patch, cmap="gray")
axes[1].set_title("Target: Expert Mask")

axes[2].imshow(image_patch, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(mask_patch == 0, mask_patch),
    cmap="autumn",
    alpha=0.6
)
axes[2].set_title("Overlay: Check Alignment")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Input shape:", image_patch.shape)
print("Target shape:", mask_patch.shape)
print("Filament pixels:", np.count_nonzero(mask_patch))

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image, ImageDraw
import numpy as np
from collections import defaultdict

class SolarPatchDataset(Dataset):
    def __init__(self, filenames, image_records, annotations,
                 image_dir, patch_size=512):

        self.image_dir = image_dir
        self.patch_size = patch_size

        self.records = [
            rec for rec in image_records
            if rec["file_name"] in filenames
        ]

        self.ann_by_id = defaultdict(list)
        for ann in annotations:
            self.ann_by_id[ann["image_id"]].append(ann)

        self.samples = []

        for rec in self.records:
            # Each annotation record is kept as its own training example
            for y in range(0, 2048, patch_size):
                for x in range(0, 2048, patch_size):
                    self.samples.append((rec, x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rec, x, y = self.samples[idx]

        # Load grayscale image
        image = Image.open(
            self.image_dir / rec["file_name"]
        ).convert("L")

        # Prepare a mask for this annotation record
        mask = Image.new("L", image.size, 0)
        draw = ImageDraw.Draw(mask)

        for ann in self.ann_by_id[rec["id"]]:
            for polygon in ann["segmentation"]:
                points = np.asarray(polygon).reshape(-1, 2)
                draw.polygon(
                    [tuple(map(float, p)) for p in points],
                    fill=1
                )

        # Crop image and mask using identical coordinates
        box = (x, y, x + self.patch_size, y + self.patch_size)

        image = np.asarray(image.crop(box), dtype=np.float32) / 255.0
        mask = np.asarray(mask.crop(box), dtype=np.float32).copy()

        # PyTorch expects: channels, height, width
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


train_dataset = SolarPatchDataset(
    filenames=train_files,
    image_records=data["images"],
    annotations=data["annotations"],
    image_dir=TRAIN_IMAGES
)

val_dataset = SolarPatchDataset(
    filenames=val_files,
    image_records=data["images"],
    annotations=data["annotations"],
    image_dir=TRAIN_IMAGES
)

print("Training patches:", len(train_dataset))
print("Validation patches:", len(val_dataset))

x, y = train_dataset[0]

print("Input shape:", x.shape)
print("Mask shape:", y.shape)
print("Image value range:", x.min().item(), x.max().item())
print("Mask unique values:", torch.unique(y))

In [ ]:
# Search for the first patch containing a filament
for i in range(len(train_dataset)):
    x, y = train_dataset[i]

    if y.sum().item() > 0:
        print("Patch index:", i)
        print("Input shape:", x.shape)
        print("Mask shape:", y.shape)
        print("Filament pixels:", int(y.sum().item()))
        print("Unique mask values:", torch.unique(y))
        break

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(x[0], cmap="gray")
axes[0].set_title("Solar Image Patch")

axes[1].imshow(y[0], cmap="gray")
axes[1].set_title("Ground-Truth Filament Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def __getitem__(self, idx):
    rec, x, y = self.samples[idx]

    box = (
        x, y,
        x + self.patch_size,
        y + self.patch_size
    )

    with Image.open(self.image_dir / rec["file_name"]) as img:
        image = img.convert("L").crop(box)

    # Create mask directly at patch resolution
    mask = Image.new(
        "L",
        (self.patch_size, self.patch_size),
        0
    )

    draw = ImageDraw.Draw(mask)

    for ann in self.ann_by_id[rec["id"]]:
        for polygon in ann["segmentation"]:
            points = np.asarray(polygon).reshape(-1, 2)

            # Convert global coordinates to patch coordinates
            local_points = [
                (float(px - x), float(py - y))
                for px, py in points
            ]

            draw.polygon(local_points, fill=1)

    image = np.asarray(image, dtype=np.float32) / 255.0
    mask = np.asarray(mask, dtype=np.float32).copy()

    image = torch.from_numpy(image).unsqueeze(0)
    mask = torch.from_numpy(mask).unsqueeze(0)

    return image, mask

In [ ]:
x, y = train_dataset[4]

print("Image shape:", x.shape)
print("Mask shape:", y.shape)
print("Filament pixels:", int(y.sum()))
print("Mask values:", torch.unique(y))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Two convolution operations
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.layers(x)


class SmallUNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(1, 16)
        self.enc2 = DoubleConv(16, 32)

        # Bottleneck: deepest feature extraction
        self.bottleneck = DoubleConv(32, 64)

        self.pool = nn.MaxPool2d(2)

        # Decoder
        self.dec2 = DoubleConv(64 + 32, 32)
        self.dec1 = DoubleConv(32 + 16, 16)

        # Produce one output channel
        self.output = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))

        # Bottleneck
        b = self.bottleneck(self.pool(e2))

        # Decoder + skip connections
        d2 = F.interpolate(
            b, size=e2.shape[2:], mode="bilinear",
            align_corners=False
        )
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = F.interpolate(
            d2, size=e1.shape[2:], mode="bilinear",
            align_corners=False
        )
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.output(d1)


model = SmallUNet()

# Test with a small input before processing 512x512 patches
test_input = torch.randn(1, 1, 128, 128)

with torch.no_grad():
    test_output = model(test_input)

print("Input shape:", test_input.shape)
print("Output shape:", test_output.shape)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

In [ ]:
import torch
import matplotlib.pyplot as plt

# Our real solar patch
x, y = train_dataset[4]

# Add batch dimension: [1, 1, 512, 512]
x_batch = x.unsqueeze(0)

model.eval()

with torch.no_grad():
    logits = model(x_batch)
    probabilities = torch.sigmoid(logits)

print("Input:", x_batch.shape)
print("Raw output:", logits.shape)
print("Probability range:",
      probabilities.min().item(),
      probabilities.max().item())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(x[0], cmap="gray")
axes[0].set_title("Original Solar Patch")

axes[1].imshow(y[0], cmap="gray")
axes[1].set_title("Expert Ground Truth")

axes[2].imshow(
    probabilities[0, 0],
    cmap="inferno",
    vmin=0,
    vmax=1
)
axes[2].set_title("Untrained U-Net Probabilities")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn.functional as F

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

# One solar image and its expert mask
x, y = train_dataset[4]

x = x.unsqueeze(0).to(device)
y = y.unsqueeze(0).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

def dice_loss(logits, targets, smooth=1.0):
    probabilities = torch.sigmoid(logits)

    intersection = (probabilities * targets).sum()

    dice = (
        2 * intersection + smooth
    ) / (
        probabilities.sum() + targets.sum() + smooth
    )

    return 1 - dice

def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(
        logits, targets
    )
    dice = dice_loss(logits, targets)

    return bce + dice

# Initial loss
model.eval()

with torch.no_grad():
    initial_logits = model(x)
    initial_loss = combined_loss(initial_logits, y)

print("Loss BEFORE training:", initial_loss.item())

# One training step
model.train()

optimizer.zero_grad()

logits = model(x)

loss = combined_loss(logits, y)

loss.backward()

optimizer.step()

# Evaluate after updating weights
model.eval()

with torch.no_grad():
    final_logits = model(x)
    final_loss = combined_loss(final_logits, y)

print("Loss AFTER training:", final_loss.item())
print("Device:", device)

In [ ]:
# Verify accelerator
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda")

# Create a fresh model
model = SmallUNet().to(device)

# Load our real solar patch
x, y = train_dataset[4]

x = x.unsqueeze(0).to(device)
y = y.unsqueeze(0).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

loss_history = []
dice_history = []

for step in range(100):

    model.train()
    optimizer.zero_grad()

    logits = model(x)

    loss = combined_loss(logits, y)

    loss.backward()
    optimizer.step()

    # Evaluate the current prediction
    model.eval()

    with torch.no_grad():
        logits = model(x)
        probabilities = torch.sigmoid(logits)

        predictions = (probabilities > 0.5).float()

        intersection = (predictions * y).sum()

        dice = (
            (2 * intersection + 1e-6) /
            (predictions.sum() + y.sum() + 1e-6)
        )

    loss_history.append(loss.item())
    dice_history.append(dice.item())

    if (step + 1) % 20 == 0:
        print(
            f"Step {step+1:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Dice: {dice.item():.4f}"
        )

# Plot training progress
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(loss_history)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Training Step")
axes[0].set_ylabel("BCE + Dice Loss")

axes[1].plot(dice_history)
axes[1].set_title("Training Dice Score")
axes[1].set_xlabel("Training Step")
axes[1].set_ylabel("Dice")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()

with torch.no_grad():
    logits = model(x)
    probs = torch.sigmoid(logits)

# Separate the true filament and background pixels
filament_probs = probs[y == 1]
background_probs = probs[y == 0]

print("Average probability on REAL filaments:",
      filament_probs.mean().item())

print("Average probability on BACKGROUND:",
      background_probs.mean().item())

print("Maximum predicted probability:",
      probs.max().item())

print("Pixels predicted as filament at 0.5:",
      int((probs > 0.5).sum().item()))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(x[0, 0].cpu(), cmap="gray")
axes[0].set_title("Original Image")

axes[1].imshow(y[0, 0].cpu(), cmap="gray")
axes[1].set_title("Expert Mask")

axes[2].imshow(
    probs[0, 0].cpu(),
    cmap="inferno",
    vmin=0,
    vmax=1
)
axes[2].set_title("U-Net After 100 Steps")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda")

# Fresh model
torch.manual_seed(42)
model = SmallUNet().to(device)

# Same image and mask
x, y = train_dataset[4]
x = x.unsqueeze(0).to(device)
y = y.unsqueeze(0).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Give filament pixels greater importance
positive_weight = torch.tensor(
    [20.0], device=device
)

def weighted_combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=positive_weight
    )

    dice = dice_loss(logits, targets)

    return bce + dice

loss_history = []
dice_history = []

for step in range(200):

    model.train()
    optimizer.zero_grad()

    logits = model(x)
    loss = weighted_combined_loss(logits, y)

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(x))
        predictions = (probs > 0.5).float()

        intersection = (predictions * y).sum()

        dice = (
            2 * intersection + 1e-6
        ) / (
            predictions.sum() + y.sum() + 1e-6
        )

    loss_history.append(loss.item())
    dice_history.append(dice.item())

    if (step + 1) % 40 == 0:
        print(
            f"Step {step+1} | "
            f"Loss: {loss.item():.4f} | "
            f"Dice: {dice.item():.4f}"
        )

# Final visual comparison
model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(x))
    final_predictions = (probs > 0.5).float()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(x[0, 0].cpu(), cmap="gray")
axes[0].set_title("Original Image")

axes[1].imshow(y[0, 0].cpu(), cmap="gray")
axes[1].set_title("Expert Mask")

axes[2].imshow(probs[0, 0].cpu(), cmap="inferno",
               vmin=0, vmax=1)
axes[2].set_title("Predicted Probabilities")

axes[3].imshow(final_predictions[0, 0].cpu(), cmap="gray")
axes[3].set_title("Predicted Binary Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Maximum probability:", probs.max().item())
print("Predicted filament pixels:",
      int(final_predictions.sum().item()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model.eval()

with torch.no_grad():
    probs = torch.sigmoid(model(x))

thresholds = [0.01, 0.03, 0.05, 0.10, 0.15, 0.20]

print(f"{'Threshold':>10} {'Dice':>10} {'Predicted pixels':>18}")

results = []

for threshold in thresholds:
    pred = (probs > threshold).float()

    tp = (pred * y).sum().item()
    fp = (pred * (1 - y)).sum().item()
    fn = ((1 - pred) * y).sum().item()

    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)

    results.append((threshold, dice, pred))

    print(
        f"{threshold:>10.2f} "
        f"{dice:>10.4f} "
        f"{int(pred.sum().item()):>18}"
    )

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, (threshold, dice, pred) in zip(axes.flat, results):
    ax.imshow(pred[0, 0].cpu(), cmap="gray")
    ax.set_title(f"Threshold={threshold:.2f} | Dice={dice:.3f}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Current x and y are GPU tensors with shape [1,1,512,512]
image_np = x[0, 0].detach().cpu().numpy()
mask_np = y[0, 0].detach().cpu().numpy()

CROP_SIZE = 256

# Find all positive pixels in the ground-truth mask
ys, xs = np.where(mask_np > 0)

# Centre crop around the filament's bounding box
cx = int((xs.min() + xs.max()) / 2)
cy = int((ys.min() + ys.max()) / 2)

x0 = int(np.clip(cx - CROP_SIZE // 2, 0, 512 - CROP_SIZE))
y0 = int(np.clip(cy - CROP_SIZE // 2, 0, 512 - CROP_SIZE))

image_crop = image_np[y0:y0+256, x0:x0+256]
mask_crop = mask_np[y0:y0+256, x0:x0+256]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(image_crop, cmap="gray")
axes[0].set_title("256×256 Solar Crop")

axes[1].imshow(mask_crop, cmap="gray")
axes[1].set_title("Expert Mask")

axes[2].imshow(image_crop, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(mask_crop == 0, mask_crop),
    cmap="autumn",
    alpha=0.6
)
axes[2].set_title("Overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Crop shape:", image_crop.shape)
print("Filament pixels:", int(mask_crop.sum()))
print("Filament percentage:",
      round(100 * mask_crop.mean(), 3), "%")

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda")

# Fresh U-Net
model = SmallUNet().to(device)

# Convert the cropped NumPy arrays to PyTorch tensors
x_small = torch.tensor(
    image_crop.copy(), dtype=torch.float32
).unsqueeze(0).unsqueeze(0).to(device)

y_small = torch.tensor(
    mask_crop.copy(), dtype=torch.float32
).unsqueeze(0).unsqueeze(0).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

pos_weight = torch.tensor([20.0], device=device)

def small_crop_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight
    )
    dice = dice_loss(logits, targets)
    return bce + dice

loss_history = []
dice_history = []

for step in range(300):
    model.train()
    optimizer.zero_grad()

    logits = model(x_small)
    loss = small_crop_loss(logits, y_small)

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(x_small))
        predictions = (probabilities > 0.5).float()

        intersection = (predictions * y_small).sum()
        dice = (
            2 * intersection + 1e-6
        ) / (
            predictions.sum() + y_small.sum() + 1e-6
        )

    loss_history.append(loss.item())
    dice_history.append(dice.item())

    if (step + 1) % 50 == 0:
        print(
            f"Step {step+1:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Dice: {dice.item():.4f}"
        )

# Final prediction
model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(x_small))
    predictions = (probabilities > 0.5).float()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(x_small[0, 0].cpu(), cmap="gray")
axes[0].set_title("Original Crop")

axes[1].imshow(y_small[0, 0].cpu(), cmap="gray")
axes[1].set_title("Expert Mask")

axes[2].imshow(
    probabilities[0, 0].cpu(),
    cmap="inferno", vmin=0, vmax=1
)
axes[2].set_title("U-Net Probabilities")

axes[3].imshow(predictions[0, 0].cpu(), cmap="gray")
axes[3].set_title("Predicted Mask")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Maximum probability:", probabilities.max().item())
print("Predicted filament pixels:", int(predictions.sum().item()))

In [ ]:
model.eval()

with torch.no_grad():
    probs = torch.sigmoid(model(x_small))
    pred = (probs > 0.5).float()

tp = ((pred == 1) & (y_small == 1)).sum().item()
fp = ((pred == 1) & (y_small == 0)).sum().item()
fn = ((pred == 0) & (y_small == 1)).sum().item()

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
iou = tp / (tp + fp + fn + 1e-8)
dice = 2 * tp / (2 * tp + fp + fn + 1e-8)

print("True Positives:", int(tp))
print("False Positives:", int(fp))
print("False Negatives:", int(fn))
print()
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"IoU:       {iou:.4f}")
print(f"Dice:      {dice:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def count_filament_pixels(dataset):
    counts = []

    # Group samples by annotation record to avoid redrawing
    # the full mask for each patch.
    for rec in dataset.records:
        full_mask = Image.new("L", (2048, 2048), 0)
        draw = ImageDraw.Draw(full_mask)

        for ann in dataset.ann_by_id[rec["id"]]:
            for polygon in ann["segmentation"]:
                points = np.asarray(polygon).reshape(-1, 2)
                draw.polygon(
                    [tuple(map(float, p)) for p in points],
                    fill=1
                )

        mask = np.asarray(full_mask)

        for y in range(0, 2048, 512):
            for x in range(0, 2048, 512):
                patch = mask[y:y+512, x:x+512]
                counts.append(int(patch.sum()))

    return np.array(counts)

train_counts = count_filament_pixels(train_dataset)

positive = np.sum(train_counts > 0)
empty = np.sum(train_counts == 0)

print("Total patches:", len(train_counts))
print("Patches containing filaments:", positive)
print("Background-only patches:", empty)

print(
    "Percentage containing filaments:",
    round(100 * positive / len(train_counts), 2), "%"
)

print(
    "Median filament pixels among positive patches:",
    np.median(train_counts[train_counts > 0])
    if positive else 0
)

plt.figure(figsize=(8, 4))
plt.hist(train_counts[train_counts > 0], bins=40)
plt.xlabel("Number of filament pixels per positive patch")
plt.ylabel("Number of patches")
plt.title("Filament Pixel Distribution in Training Patches")
plt.show()

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

# train_counts follows the same record/patch ordering
# as train_dataset.samples

assert len(train_counts) == len(train_dataset)

positive_mask = train_counts > 0

n_positive = positive_mask.sum()
n_negative = (~positive_mask).sum()

weights = np.where(
    positive_mask,
    0.5 / n_positive,
    0.5 / n_negative
)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

images, masks = next(iter(train_loader))

print("Batch images:", images.shape)
print("Batch masks:", masks.shape)

print(
    "Positive patches in this batch:",
    int((masks.sum(dim=(1, 2, 3)) > 0).sum())
)

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda")

# Fresh model
torch.manual_seed(42)
model = SmallUNet().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

positive_weight = torch.tensor([20.0], device=device)

def training_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=positive_weight
    )

    dice = dice_loss(logits, targets)

    return bce + dice


def calculate_metrics(logits, targets):
    probs = torch.sigmoid(logits)
    pred = (probs > 0.5).float()

    tp = (pred * targets).sum().item()
    fp = (pred * (1 - targets)).sum().item()
    fn = ((1 - pred) * targets).sum().item()

    return tp, fp, fn


train_losses = []
val_dice_scores = []
val_iou_scores = []

EPOCHS = 3
MAX_TRAIN_BATCHES = 200
MAX_VAL_BATCHES = 100

for epoch in range(EPOCHS):

    # TRAINING
    model.train()
    running_loss = 0
    train_batches = 0

    for images, masks in train_loader:

        if train_batches >= MAX_TRAIN_BATCHES:
            break

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images)
        loss = training_loss(logits, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        train_batches += 1

    average_train_loss = running_loss / train_batches

    # VALIDATION
    model.eval()

    total_tp = 0
    total_fp = 0
    total_fn = 0

    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(val_loader):

            if batch_idx >= MAX_VAL_BATCHES:
                break

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            logits = model(images)

            tp, fp, fn = calculate_metrics(logits, masks)

            total_tp += tp
            total_fp += fp
            total_fn += fn

    val_dice = (
        2 * total_tp /
        (2 * total_tp + total_fp + total_fn + 1e-8)
    )

    val_iou = (
        total_tp /
        (total_tp + total_fp + total_fn + 1e-8)
    )

    train_losses.append(average_train_loss)
    val_dice_scores.append(val_dice)
    val_iou_scores.append(val_iou)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {average_train_loss:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Val IoU: {val_iou:.4f}"
    )

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(1, EPOCHS + 1), train_losses, marker="o")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(range(1, EPOCHS + 1), val_dice_scores,
             marker="o", label="Dice")
axes[1].plot(range(1, EPOCHS + 1), val_iou_scores,
             marker="o", label="IoU")
axes[1].set_title("Validation Performance")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()

# Find a validation patch containing annotated filaments
for i in range(len(val_dataset)):
    val_x, val_y = val_dataset[i]

    if val_y.sum().item() > 500:
        break

# Move to GPU
input_batch = val_x.unsqueeze(0).to(device)
target_batch = val_y.unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_batch)
    probs = torch.sigmoid(logits)

filament_probs = probs[target_batch == 1]
background_probs = probs[target_batch == 0]

print("Validation patch index:", i)
print("Expert filament pixels:", int(target_batch.sum().item()))
print("Average filament probability:", filament_probs.mean().item())
print("Average background probability:", background_probs.mean().item())
print("Maximum probability:", probs.max().item())
print("Predicted pixels at 0.5:", int((probs > 0.5).sum().item()))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(val_x[0], cmap="gray")
axes[0].set_title("Unseen Solar Image")

axes[1].imshow(val_y[0], cmap="gray")
axes[1].set_title("Expert Mask")

axes[2].imshow(
    probs[0, 0].cpu(),
    cmap="inferno",
    vmin=0,
    vmax=1
)
axes[2].set_title("U-Net Prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from PIL import Image, ImageDraw

rng = np.random.default_rng(42)

# Identify positive/background patches using metadata
val_counts = count_filament_pixels(val_dataset)

positive_indices = np.flatnonzero(val_counts > 0)
negative_indices = np.flatnonzero(val_counts == 0)

print("Positive validation patches:", len(positive_indices))
print("Background-only validation patches:", len(negative_indices))

# Select up to 50 of each
selected_positive = rng.choice(
    positive_indices,
    size=min(50, len(positive_indices)),
    replace=False
)

selected_negative = rng.choice(
    negative_indices,
    size=min(50, len(negative_indices)),
    replace=False
)

selected_indices = np.concatenate([
    selected_positive,
    selected_negative
])

validation_subset = Subset(
    val_dataset,
    selected_indices.tolist()
)

validation_loader = DataLoader(
    validation_subset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Evaluate
model.eval()

total_tp = total_fp = total_fn = 0

with torch.no_grad():
    for images, masks in validation_loader:

        images = images.to(device)
        masks = masks.to(device)

        probabilities = torch.sigmoid(model(images))
        predictions = probabilities > 0.5

        total_tp += ((predictions == 1) & (masks == 1)).sum().item()
        total_fp += ((predictions == 1) & (masks == 0)).sum().item()
        total_fn += ((predictions == 0) & (masks == 1)).sum().item()

dice = 2 * total_tp / (2 * total_tp + total_fp + total_fn + 1e-8)
iou = total_tp / (total_tp + total_fp + total_fn + 1e-8)

print("\nValidation sample results")
print("TP:", int(total_tp))
print("FP:", int(total_fp))
print("FN:", int(total_fn))
print(f"Dice: {dice:.4f}")
print(f"IoU: {iou:.4f}")

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw

class SolarFocusedDataset(Dataset):
    def __init__(self, records, annotations_by_id, image_dir,
                 patch_size=256, seed=42):

        self.records = records
        self.ann_by_id = annotations_by_id
        self.image_dir = image_dir
        self.patch_size = patch_size
        self.rng_seed = seed

        # One filament-centred and one random crop per record
        self.samples = []

        for rec in records:
            if len(self.ann_by_id[rec["id"]]) > 0:
                self.samples.append((rec, "filament"))

            self.samples.append((rec, "random"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rec, crop_type = self.samples[idx]
        size = self.patch_size

        # Reproducible random choices for this sample
        rng = np.random.default_rng(self.rng_seed + idx)

        with Image.open(self.image_dir / rec["file_name"]) as img:
            image = img.convert("L")
            width, height = image.size

            if crop_type == "filament":
                anns = self.ann_by_id[rec["id"]]
                ann = anns[int(rng.integers(len(anns)))]

                # Bounding-box centre
                bx, by, bw, bh = ann["bbox"]
                cx = bx + bw / 2
                cy = by + bh / 2

                x0 = int(np.clip(
                    cx - size / 2, 0, width - size
                ))
                y0 = int(np.clip(
                    cy - size / 2, 0, height - size
                ))

            else:
                x0 = int(rng.integers(0, width - size + 1))
                y0 = int(rng.integers(0, height - size + 1))

            image_crop = image.crop(
                (x0, y0, x0 + size, y0 + size)
            )

        # Draw expert polygons in local crop coordinates
        mask = Image.new("L", (size, size), 0)
        draw = ImageDraw.Draw(mask)

        for ann in self.ann_by_id[rec["id"]]:
            for polygon in ann["segmentation"]:
                points = np.asarray(polygon).reshape(-1, 2)

                local_points = [
                    (float(px - x0), float(py - y0))
                    for px, py in points
                ]

                draw.polygon(local_points, fill=1)

        x = torch.from_numpy(
            np.asarray(image_crop, dtype=np.float32) / 255.0
        ).unsqueeze(0)

        y = torch.from_numpy(
            np.asarray(mask, dtype=np.float32).copy()
        ).unsqueeze(0)

        return x, y


focused_train = SolarFocusedDataset(
    records=train_records,
    annotations_by_id=train_dataset.ann_by_id,
    image_dir=TRAIN_IMAGES
)

print("Focused training samples:", len(focused_train))

for i in range(4):
    x, y = focused_train[i]

    print(
        f"Sample {i}:",
        "shape =", tuple(x.shape),
        "filament pixels =", int(y.sum())
    )

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

device = torch.device("cuda")

# DataLoader for our new focused dataset
focused_loader = DataLoader(
    focused_train,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Create a fresh U-Net
torch.manual_seed(42)
model = SmallUNet().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005
)

pos_weight = torch.tensor([20.0], device=device)

def focused_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=pos_weight
    )
    return bce + dice_loss(logits, targets)

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0.0
    total_tp = 0
    total_fp = 0
    total_fn = 0

    for images, masks in focused_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images)
        loss = focused_loss(logits, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        with torch.no_grad():
            predictions = torch.sigmoid(logits) > 0.5
            positives = masks > 0.5

            total_tp += (predictions & positives).sum().item()
            total_fp += (predictions & ~positives).sum().item()
            total_fn += (~predictions & positives).sum().item()

    average_loss = total_loss / len(focused_loader)

    train_dice = (
        2 * total_tp /
        (2 * total_tp + total_fp + total_fn + 1e-8)
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {average_loss:.4f} | "
        f"Training Dice: {train_dice:.4f}"
    )

# Save the trained model
torch.save(
    model.state_dict(),
    "/kaggle/working/solar_unet_baseline.pth"
)

print("Model checkpoint saved.")

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval()

# First filament-focused training sample
x, y = focused_train[0]

x = x.unsqueeze(0).to(device)
y = y.unsqueeze(0).to(device)

with torch.no_grad():
    probs = torch.sigmoid(model(x))

filament_probs = probs[y == 1]
background_probs = probs[y == 0]

print("Average filament probability:",
      filament_probs.mean().item())

print("Average background probability:",
      background_probs.mean().item())

print("Maximum probability:",
      probs.max().item())

print("Predicted pixels at threshold 0.5:",
      int((probs > 0.5).sum().item()))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(x[0, 0].cpu(), cmap="gray")
axes[0].set_title("Solar Image")

axes[1].imshow(y[0, 0].cpu(), cmap="gray")
axes[1].set_title("Expert Mask")

axes[2].imshow(
    probs[0, 0].cpu(),
    cmap="inferno",
    vmin=0,
    vmax=1
)
axes[2].set_title("Trained U-Net Probabilities")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# Select a reproducible mixture of validation patches
rng = np.random.default_rng(42)

positive_indices = np.flatnonzero(val_counts > 0)
negative_indices = np.flatnonzero(val_counts == 0)

selected = np.concatenate([
    rng.choice(positive_indices, min(50, len(positive_indices)),
               replace=False),
    rng.choice(negative_indices, min(50, len(negative_indices)),
               replace=False)
])

loader = DataLoader(
    Subset(val_dataset, selected.tolist()),
    batch_size=4,
    shuffle=False,
    num_workers=2
)

thresholds = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50]

tp = np.zeros(len(thresholds))
fp = np.zeros(len(thresholds))
fn = np.zeros(len(thresholds))

model.eval()

with torch.no_grad():
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        probabilities = torch.sigmoid(model(images))

        for j, threshold in enumerate(thresholds):
            pred = probabilities > threshold
            truth = masks > 0.5

            tp[j] += (pred & truth).sum().item()
            fp[j] += (pred & ~truth).sum().item()
            fn[j] += (~pred & truth).sum().item()

print(f"{'Threshold':>10} {'Dice':>10} {'IoU':>10}")

for j, threshold in enumerate(thresholds):
    dice = 2 * tp[j] / (2 * tp[j] + fp[j] + fn[j] + 1e-8)
    iou = tp[j] / (tp[j] + fp[j] + fn[j] + 1e-8)

    print(f"{threshold:>10.2f} {dice:>10.4f} {iou:>10.4f}")

In [ ]:
# Create validation data with the same 256x256 crop strategy
focused_val = SolarFocusedDataset(
    records=val_records,
    annotations_by_id=val_dataset.ann_by_id,
    image_dir=TRAIN_IMAGES,
    patch_size=256,
    seed=123
)

focused_val_loader = DataLoader(
    focused_val,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

counts = {
    t: {"tp": 0, "fp": 0, "fn": 0}
    for t in thresholds
}

model.eval()

with torch.no_grad():
    for images, masks in focused_val_loader:
        images = images.to(device)
        masks = masks.to(device)

        probs = torch.sigmoid(model(images))
        truth = masks > 0.5

        for t in thresholds:
            pred = probs > t

            counts[t]["tp"] += (pred & truth).sum().item()
            counts[t]["fp"] += (pred & ~truth).sum().item()
            counts[t]["fn"] += (~pred & truth).sum().item()

print("Validation image records:", len(val_records))
print("Validation crops:", len(focused_val))

print(f"{'Threshold':>10} {'Dice':>10} {'IoU':>10}")

for t in thresholds:
    tp = counts[t]["tp"]
    fp = counts[t]["fp"]
    fn = counts[t]["fn"]

    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)

    print(f"{t:10.2f} {dice:10.4f} {iou:10.4f}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

model.eval()

# Select an unseen validation photograph
filename = sorted(val_files)[0]

solar_image = np.asarray(
    Image.open(TRAIN_IMAGES / filename).convert("L"),
    dtype=np.float32
) / 255.0

PATCH = 256
H, W = solar_image.shape

probability_map = np.zeros((H, W), dtype=np.float32)

with torch.no_grad():
    for y0 in range(0, H, PATCH):
        for x0 in range(0, W, PATCH):

            patch = solar_image[
                y0:y0+PATCH,
                x0:x0+PATCH
            ]

            tensor = torch.from_numpy(
                patch.copy()
            ).unsqueeze(0).unsqueeze(0).to(device)

            probs = torch.sigmoid(model(tensor))

            probability_map[
                y0:y0+PATCH,
                x0:x0+PATCH
            ] = probs[0, 0].cpu().numpy()

binary_prediction = probability_map > 0.45

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(solar_image, cmap="gray")
axes[0].set_title("Original Solar Image")

axes[1].imshow(
    probability_map,
    cmap="inferno",
    vmin=0,
    vmax=1
)
axes[1].set_title("U-Net Probability Map")

axes[2].imshow(binary_prediction, cmap="gray")
axes[2].set_title("Predicted Mask (Threshold 0.45)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image:", filename)
print("Original shape:", solar_image.shape)
print("Reconstructed shape:", probability_map.shape)
print("Maximum probability:", probability_map.max())
print("Predicted filament pixels:", binary_prediction.sum())

In [ ]:
import torch

ADDITIONAL_EPOCHS = 5
THRESHOLD = 0.45

best_val_dice = -1.0

for epoch in range(ADDITIONAL_EPOCHS):

    model.train()
    running_loss = 0.0

    for images, masks in focused_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = focused_loss(logits, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Validation
    model.eval()
    tp = fp = fn = 0

    with torch.no_grad():
        for images, masks in focused_val_loader:
            images = images.to(device)
            masks = masks.to(device)

            pred = torch.sigmoid(model(images)) > THRESHOLD
            truth = masks > 0.5

            tp += (pred & truth).sum().item()
            fp += (pred & ~truth).sum().item()
            fn += (~pred & truth).sum().item()

    val_dice = 2 * tp / (2 * tp + fp + fn + 1e-8)

    print(
        f"Additional epoch {epoch+1}/{ADDITIONAL_EPOCHS} | "
        f"Train Loss: {running_loss/len(focused_loader):.4f} | "
        f"Val Dice: {val_dice:.4f}"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(
            model.state_dict(),
            "/kaggle/working/solar_unet_best.pth"
        )

print("Best additional-training validation Dice:", best_val_dice)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device("cuda")

# Load the best validation checkpoint
best_model = SmallUNet().to(device)

best_model.load_state_dict(
    torch.load(
        "/kaggle/working/solar_unet_best.pth",
        map_location=device,
        weights_only=True
    )
)

best_model.eval()

# Use the same validation photograph as before
filename = sorted(val_files)[0]

solar_image = np.asarray(
    Image.open(TRAIN_IMAGES / filename).convert("L"),
    dtype=np.float32
) / 255.0

PATCH = 256
H, W = solar_image.shape

probability_map = np.zeros((H, W), dtype=np.float32)

# Predict the full image using 256x256 patches
with torch.inference_mode():
    for y0 in range(0, H, PATCH):
        for x0 in range(0, W, PATCH):

            patch = solar_image[
                y0:y0+PATCH,
                x0:x0+PATCH
            ]

            tensor = torch.from_numpy(
                patch.copy()
            ).unsqueeze(0).unsqueeze(0).to(device)

            probabilities = torch.sigmoid(
                best_model(tensor)
            )

            probability_map[
                y0:y0+PATCH,
                x0:x0+PATCH
            ] = probabilities[0, 0].cpu().numpy()

prediction = probability_map > 0.45

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(solar_image, cmap="gray")
axes[0].set_title("Original Solar Image")

axes[1].imshow(
    probability_map,
    cmap="inferno",
    vmin=0,
    vmax=1
)
axes[1].set_title("Best U-Net Probability Map")

axes[2].imshow(prediction, cmap="gray")
axes[2].set_title("Predicted Filaments (0.45)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image:", filename)
print("Maximum probability:", probability_map.max())
print("Predicted filament pixels:", int(prediction.sum()))

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

PATCH = 256
STRIDE = 128

H, W = solar_image.shape

prob_sum = np.zeros((H, W), dtype=np.float32)
weight_sum = np.zeros((H, W), dtype=np.float32)

best_model.eval()

# Batch several patches together for faster GPU inference
patches = []
positions = []

for y0 in range(0, H - PATCH + 1, STRIDE):
    for x0 in range(0, W - PATCH + 1, STRIDE):
        patch = solar_image[y0:y0+PATCH, x0:x0+PATCH]

        patches.append(patch)
        positions.append((y0, x0))

BATCH_SIZE = 8

with torch.inference_mode():
    for start in range(0, len(patches), BATCH_SIZE):

        batch = np.stack(
            patches[start:start+BATCH_SIZE]
        )

        tensor = torch.from_numpy(
            batch.copy()
        ).unsqueeze(1).to(device)

        probabilities = torch.sigmoid(
            best_model(tensor)
        ).cpu().numpy()[:, 0]

        for pred, (y0, x0) in zip(
            probabilities,
            positions[start:start+BATCH_SIZE]
        ):
            prob_sum[
                y0:y0+PATCH,
                x0:x0+PATCH
            ] += pred

            weight_sum[
                y0:y0+PATCH,
                x0:x0+PATCH
            ] += 1

# Average overlapping predictions
assert np.all(weight_sum > 0)

overlap_map = prob_sum / weight_sum
overlap_mask = overlap_map > 0.45

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(solar_image, cmap="gray")
axes[0].set_title("Original Solar Image")

axes[1].imshow(
    overlap_map,
    cmap="inferno",
    vmin=0,
    vmax=1
)
axes[1].set_title("Overlapping Probability Map")

axes[2].imshow(overlap_mask, cmap="gray")
axes[2].set_title("Overlapping Prediction")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Number of patches:", len(patches))
print("Prediction shape:", overlap_map.shape)
print("Predicted filament pixels:", int(overlap_mask.sum()))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Binary mask from overlapping U-Net inference
binary_mask = overlap_mask.astype(np.uint8)

# Find connected regions (8-neighbour connectivity)
num_labels, labels, stats, centroids = (
    cv2.connectedComponentsWithStats(
        binary_mask,
        connectivity=8
    )
)

# Ignore components smaller than 50 pixels
MIN_AREA = 50

instances = []

for component_id in range(1, num_labels):
    area = stats[component_id, cv2.CC_STAT_AREA]

    if area < MIN_AREA:
        continue

    instance_mask = labels == component_id
    instances.append(instance_mask)

print("All foreground components:", num_labels - 1)
print("Components retained:", len(instances))

# Show original mask and retained components
filtered_mask = np.zeros_like(binary_mask)

for instance in instances:
    filtered_mask[instance] = 1

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(solar_image, cmap="gray")
axes[0].set_title("Original Solar Image")

axes[1].imshow(binary_mask, cmap="gray")
axes[1].set_title("Raw U-Net Prediction")

axes[2].imshow(filtered_mask, cmap="gray")
axes[2].set_title("Retained Components (Area ≥ 50)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Pixels before filtering:", int(binary_mask.sum()))
print("Pixels after filtering:", int(filtered_mask.sum()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

areas = np.array([
    stats[i, cv2.CC_STAT_AREA]
    for i in range(1, num_labels)
])

print("Total components:", len(areas))
print("Median area:", np.median(areas))
print("90th percentile:", np.percentile(areas, 90))
print("Maximum area:", areas.max())

for minimum in [50, 100, 200, 400, 800]:
    count = np.sum(areas >= minimum)
    print(f"Area >= {minimum:4d}: {count} components")

plt.figure(figsize=(8, 4))
plt.hist(areas[areas >= 50], bins=40)
plt.xlabel("Component area (pixels)")
plt.ylabel("Number of components")
plt.title("Predicted Filament Candidate Sizes")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

MIN_AREA = 200

num_labels, labels, stats, centroids = (
    cv2.connectedComponentsWithStats(
        overlap_mask.astype(np.uint8),
        connectivity=8
    )
)

# Create a colour visualisation
coloured = np.zeros(
    (2048, 2048, 3), dtype=np.uint8
)

rng = np.random.default_rng(42)

retained = []

for component_id in range(1, num_labels):
    area = stats[component_id, cv2.CC_STAT_AREA]

    if area < MIN_AREA:
        continue

    instance_mask = labels == component_id
    retained.append(component_id)

    colour = rng.integers(80, 256, size=3, dtype=np.uint8)
    coloured[instance_mask] = colour

print("Individual candidate filaments:", len(retained))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

axes[0].imshow(solar_image, cmap="gray")
axes[0].set_title("Original Solar Image")

axes[1].imshow(coloured)
axes[1].set_title("Individual Candidate Filaments")

for component_id in retained:
    x, y, w, h, area = stats[component_id]

    rect = plt.Rectangle(
        (x, y), w, h,
        fill=False,
        edgecolor="yellow",
        linewidth=0.6
    )

    axes[1].add_patch(rect)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image

try:
    from pycocotools import mask as mask_utils
except ImportError:
    %pip install -q pycocotools
    from pycocotools import mask as mask_utils

PATCH = 256
STRIDE = 128
THRESHOLD = 0.45
MIN_AREA = 200
BATCH_SIZE = 8

test_files = sorted(TEST_IMAGES.glob("*.jpeg"))

print("Test images:", len(test_files))
assert len(test_files) == 180

best_model.eval()

def predict_full_image(image_path):
    with Image.open(image_path) as im:
        image = np.asarray(
            im.convert("L"), dtype=np.float32
        ) / 255.0

    H, W = image.shape

    prob_sum = np.zeros((H, W), dtype=np.float32)
    weight_sum = np.zeros((H, W), dtype=np.float32)

    positions = [
        (y, x)
        for y in range(0, H - PATCH + 1, STRIDE)
        for x in range(0, W - PATCH + 1, STRIDE)
    ]

    with torch.inference_mode():
        for start in range(0, len(positions), BATCH_SIZE):
            current = positions[start:start+BATCH_SIZE]

            batch = np.stack([
                image[y:y+PATCH, x:x+PATCH]
                for y, x in current
            ])

            tensor = torch.from_numpy(
                batch.copy()
            ).unsqueeze(1).to(device)

            probs = torch.sigmoid(
                best_model(tensor)
            ).cpu().numpy()[:, 0]

            for pred, (y, x) in zip(probs, current):
                prob_sum[y:y+PATCH, x:x+PATCH] += pred
                weight_sum[y:y+PATCH, x:x+PATCH] += 1

    assert np.all(weight_sum > 0)

    return prob_sum / weight_sum


submission_rows = []
counts_per_image = {}

for image_number, image_path in enumerate(test_files, start=1):

    probability_map = predict_full_image(image_path)

    binary_mask = (
        probability_map > THRESHOLD
    ).astype(np.uint8)

    num_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            binary_mask, connectivity=8
        )
    )

    filament_number = 0
    image_id = image_path.stem

    for component_id in range(1, num_labels):
        area = stats[component_id, cv2.CC_STAT_AREA]

        if area < MIN_AREA:
            continue

        # One mask for one filament
        instance_mask = np.asfortranarray(
            (labels == component_id).astype(np.uint8)
        )

        encoded = mask_utils.encode(instance_mask)
        rle_counts = encoded["counts"].decode("ascii")

        filament_number += 1

        submission_rows.append({
            "filament_id": f"{image_id}_{filament_number}",
            "segmentation_rle": rle_counts
        })

    counts_per_image[image_id] = filament_number

    if image_number % 20 == 0:
        print(
            f"Processed {image_number}/{len(test_files)} images"
        )

submission = pd.DataFrame(
    submission_rows,
    columns=["filament_id", "segmentation_rle"]
)

output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)

print("\nSubmission created:", output_path)
print("Total predicted filaments:", len(submission))
print("Images processed:", len(counts_per_image))
print("Images with no predicted filaments:",
      sum(n == 0 for n in counts_per_image.values()))

print("\nFirst five rows:")
print(submission.head())

In [ ]:
import pandas as pd
import numpy as np
from pycocotools import mask as mask_utils

submission = pd.read_csv(
    "/kaggle/working/submission.csv",
    dtype=str
)

assert list(submission.columns) == [
    "filament_id", "segmentation_rle"
]

assert submission["filament_id"].is_unique
assert submission["segmentation_rle"].notna().all()
assert (submission["segmentation_rle"].str.len() > 0).all()

test_ids = {
    path.stem for path in TEST_IMAGES.glob("*.jpeg")
}

predicted_ids = {
    fid.rsplit("_", 1)[0]
    for fid in submission["filament_id"]
}

assert predicted_ids.issubset(test_ids)

# Decode and verify 5 masks
for i in range(min(5, len(submission))):
    rle = {
        "size": [2048, 2048],
        "counts": submission.iloc[i]["segmentation_rle"].encode("ascii")
    }

    decoded = mask_utils.decode(rle)

    assert decoded.shape == (2048, 2048)
    assert decoded.sum() > 0

    print(
        submission.iloc[i]["filament_id"],
        "Decoded pixels:",
        int(decoded.sum())
    )

print("\nCSV validation passed!")
print("Rows:", len(submission))
print("Unique images with predictions:", len(predicted_ids))
print("Images without predictions:", len(test_ids - predicted_ids))

In [ ]:

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

THRESHOLDS = [0.40, 0.45, 0.50, 0.55]
MIN_AREAS = [50, 100, 200, 400, 800]
IOU_MATCH = 0.50

# Build individual expert masks for a single annotated image
def get_ground_truth(filename):
    rec = records_by_filename[filename]
    anns = annotations_by_id.get(rec["id"], [])
    h, w = int(rec["height"]), int(rec["width"])

    masks = []
    for ann in anns:
        mask = Image.new("1", (w, h), 0)
        draw = ImageDraw.Draw(mask)

        for polygon in ann["segmentation"]:
            points = np.asarray(polygon).reshape(-1, 2)
            draw.polygon(
                [tuple(map(float, p)) for p in points],
                fill=1
            )

        arr = np.asarray(mask, dtype=bool)
        if arr.any():
            masks.append(arr)

    return masks


def evaluate_image(probability_map, gt_masks, threshold, min_area):
    binary = (probability_map > threshold).astype(np.uint8)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    # Candidate predicted components and their areas
    ids = [
        i for i in range(1, n)
        if stats[i, cv2.CC_STAT_AREA] >= min_area
    ]

    pred_areas = {
        i: int(stats[i, cv2.CC_STAT_AREA]) for i in ids
    }

    # Find candidate matches using IoU
    candidates = []

    for j, gt in enumerate(gt_masks):
        gt_area = int(gt.sum())
        if gt_area == 0:
            continue

        # Count overlap of all predicted components with this GT
        intersect_counts = np.bincount(
            labels[gt].ravel(), minlength=n
        )

        for i in ids:
            intersection = int(intersect_counts[i])

            if intersection == 0:
                continue

            union = pred_areas[i] + gt_area - intersection
            iou = intersection / union

            if iou > IOU_MATCH:
                candidates.append((iou, i, j))

    # Greedy one-to-one matching, highest IoU first
    candidates.sort(reverse=True)

    used_predictions = set()
    used_truths = set()
    sum_iou = 0.0

    for iou, pred_id, gt_id in candidates:
        if pred_id in used_predictions or gt_id in used_truths:
            continue

        used_predictions.add(pred_id)
        used_truths.add(gt_id)
        sum_iou += iou

    tp = len(used_predictions)
    fp = len(ids) - tp
    fn = len(gt_masks) - tp

    return tp, fp, fn, sum_iou


# Cache the reference masks once
ground_truth = {
    name: get_ground_truth(name) for name in pilot_files
}

probability_maps = {
    name: np.load(CACHE_DIR / (Path(name).stem + ".npy"))
    for name in pilot_files
}

results = []

for threshold in THRESHOLDS:
    for min_area in MIN_AREAS:
        TP = FP = FN = 0
        total_iou = 0.0

        for filename in pilot_files:
            tp, fp, fn, matched_iou = evaluate_image(
                probability_maps[filename],
                ground_truth[filename],
                threshold,
                min_area
            )

            TP += tp
            FP += fp
            FN += fn
            total_iou += matched_iou

        denominator = TP + 0.5 * FP + 0.5 * FN

        pq = total_iou / denominator if denominator else 0.0

        results.append({
            "threshold": threshold,
            "min_area": min_area,
            "PQ_proxy": pq,
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "predicted_instances": TP + FP
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "PQ_proxy", ascending=False
).reset_index(drop=True)

# V1 reference configuration
baseline = results_df[
    (results_df["threshold"] == 0.45) &
    (results_df["min_area"] == 200)
].iloc[0]

print("V1 baseline on these 12 images:")
print(baseline.to_string())

print("\nTop 10 settings:")
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]

print("\nBest pilot configuration:")
print(f"Threshold: {best['threshold']:.2f}")
print(f"Minimum area: {int(best['min_area'])}")
print(f"PQ proxy: {best['PQ_proxy']:.4f}")
print(f"V1 PQ proxy: {baseline['PQ_proxy']:.4f}")

results_df.to_csv(
    "/kaggle/working/v2_pilot_results.csv",
    index=False
)


In [ ]:

import numpy as np
import torch
from pathlib import Path
from PIL import Image

# Experiment configuration
SEED = 42
N_PILOT = 12
PATCH = 256
STRIDE = 128
BATCH_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Restore our best V1 model
best_model = SmallUNet().to(device)
best_model.load_state_dict(
    torch.load(
        "/kaggle/working/solar_unet_best.pth",
        map_location=device,
        weights_only=True
    )
)
best_model.eval()

# Choose 12 held-out physical photographs
rng = np.random.default_rng(SEED)
pilot_files = sorted(
    rng.choice(sorted(val_files), size=N_PILOT, replace=False).tolist()
)

# Use one annotation record for each photograph.
# This prevents multiple annotations of one photograph from
# being counted as different validation photographs.
records_by_filename = {}
for rec in sorted(data["images"], key=lambda r: str(r["id"])):
    records_by_filename.setdefault(rec["file_name"], rec)

annotations_by_id = {}
for ann in data["annotations"]:
    annotations_by_id.setdefault(ann["image_id"], []).append(ann)

CACHE_DIR = Path("/kaggle/working/v2_pilot_cache")
CACHE_DIR.mkdir(exist_ok=True)

def predict_full(image_path):
    with Image.open(image_path) as im:
        image = np.asarray(im.convert("L"), dtype=np.float32) / 255.0

    height, width = image.shape

    ys = list(range(0, height - PATCH + 1, STRIDE))
    xs = list(range(0, width - PATCH + 1, STRIDE))

    positions = [(y, x) for y in ys for x in xs]

    prob_sum = np.zeros((height, width), dtype=np.float32)
    weight_sum = np.zeros((height, width), dtype=np.float32)

    with torch.inference_mode():
        for start in range(0, len(positions), BATCH_SIZE):
            batch_positions = positions[start:start+BATCH_SIZE]

            batch = np.stack([
                image[y:y+PATCH, x:x+PATCH]
                for y, x in batch_positions
            ])

            tensor = torch.from_numpy(batch.copy()).unsqueeze(1).to(device)

            probabilities = torch.sigmoid(
                best_model(tensor)
            ).cpu().numpy()[:, 0]

            for prediction, (y, x) in zip(probabilities, batch_positions):
                prob_sum[y:y+PATCH, x:x+PATCH] += prediction
                weight_sum[y:y+PATCH, x:x+PATCH] += 1

    assert np.all(weight_sum > 0)
    return prob_sum / weight_sum

for index, filename in enumerate(pilot_files, 1):
    destination = CACHE_DIR / (Path(filename).stem + ".npy")

    if not destination.exists():
        prob = predict_full(TRAIN_IMAGES / filename)
        np.save(destination, prob)

    print(f"{index}/{N_PILOT}: {filename}")

print("\nStage 1 complete.")
print("Cached probability maps:", len(list(CACHE_DIR.glob("*.npy"))))


In [ ]:

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

THRESHOLDS = [0.40, 0.45, 0.50, 0.55]
MIN_AREAS = [50, 100, 200, 400, 800]
IOU_MATCH = 0.50

# Build individual expert masks for a single annotated image
def get_ground_truth(filename):
    rec = records_by_filename[filename]
    anns = annotations_by_id.get(rec["id"], [])
    h, w = int(rec["height"]), int(rec["width"])

    masks = []
    for ann in anns:
        mask = Image.new("1", (w, h), 0)
        draw = ImageDraw.Draw(mask)

        for polygon in ann["segmentation"]:
            points = np.asarray(polygon).reshape(-1, 2)
            draw.polygon(
                [tuple(map(float, p)) for p in points],
                fill=1
            )

        arr = np.asarray(mask, dtype=bool)
        if arr.any():
            masks.append(arr)

    return masks


def evaluate_image(probability_map, gt_masks, threshold, min_area):
    binary = (probability_map > threshold).astype(np.uint8)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )

    # Candidate predicted components and their areas
    ids = [
        i for i in range(1, n)
        if stats[i, cv2.CC_STAT_AREA] >= min_area
    ]

    pred_areas = {
        i: int(stats[i, cv2.CC_STAT_AREA]) for i in ids
    }

    # Find candidate matches using IoU
    candidates = []

    for j, gt in enumerate(gt_masks):
        gt_area = int(gt.sum())
        if gt_area == 0:
            continue

        # Count overlap of all predicted components with this GT
        intersect_counts = np.bincount(
            labels[gt].ravel(), minlength=n
        )

        for i in ids:
            intersection = int(intersect_counts[i])

            if intersection == 0:
                continue

            union = pred_areas[i] + gt_area - intersection
            iou = intersection / union

            if iou > IOU_MATCH:
                candidates.append((iou, i, j))

    # Greedy one-to-one matching, highest IoU first
    candidates.sort(reverse=True)

    used_predictions = set()
    used_truths = set()
    sum_iou = 0.0

    for iou, pred_id, gt_id in candidates:
        if pred_id in used_predictions or gt_id in used_truths:
            continue

        used_predictions.add(pred_id)
        used_truths.add(gt_id)
        sum_iou += iou

    tp = len(used_predictions)
    fp = len(ids) - tp
    fn = len(gt_masks) - tp

    return tp, fp, fn, sum_iou


# Cache the reference masks once
ground_truth = {
    name: get_ground_truth(name) for name in pilot_files
}

probability_maps = {
    name: np.load(CACHE_DIR / (Path(name).stem + ".npy"))
    for name in pilot_files
}

results = []

for threshold in THRESHOLDS:
    for min_area in MIN_AREAS:
        TP = FP = FN = 0
        total_iou = 0.0

        for filename in pilot_files:
            tp, fp, fn, matched_iou = evaluate_image(
                probability_maps[filename],
                ground_truth[filename],
                threshold,
                min_area
            )

            TP += tp
            FP += fp
            FN += fn
            total_iou += matched_iou

        denominator = TP + 0.5 * FP + 0.5 * FN

        pq = total_iou / denominator if denominator else 0.0

        results.append({
            "threshold": threshold,
            "min_area": min_area,
            "PQ_proxy": pq,
            "TP": TP,
            "FP": FP,
            "FN": FN,
            "predicted_instances": TP + FP
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "PQ_proxy", ascending=False
).reset_index(drop=True)

# V1 reference configuration
baseline = results_df[
    (results_df["threshold"] == 0.45) &
    (results_df["min_area"] == 200)
].iloc[0]

print("V1 baseline on these 12 images:")
print(baseline.to_string())

print("\nTop 10 settings:")
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]

print("\nBest pilot configuration:")
print(f"Threshold: {best['threshold']:.2f}")
print(f"Minimum area: {int(best['min_area'])}")
print(f"PQ proxy: {best['PQ_proxy']:.4f}")
print(f"V1 PQ proxy: {baseline['PQ_proxy']:.4f}")

results_df.to_csv(
    "/kaggle/working/v2_pilot_results.csv",
    index=False
)


In [ ]:

import matplotlib.pyplot as plt

pivot = results_df.pivot(
    index="min_area",
    columns="threshold",
    values="PQ_proxy"
)

fig, ax = plt.subplots(figsize=(8, 5))

heat = ax.imshow(pivot.values, aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{v:.2f}" for v in pivot.columns])

ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

ax.set_xlabel("Probability threshold")
ax.set_ylabel("Minimum component area")
ax.set_title("V2 Pilot — Instance PQ Proxy")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(
            j, i,
            f"{pivot.values[i, j]:.3f}",
            ha="center", va="center",
            fontsize=9,
            color="white" if pivot.values[i,j] <
            (np.nanmin(pivot.values)+np.nanmax(pivot.values))/2
            else "black"
        )

plt.colorbar(heat, ax=ax, label="PQ proxy")
plt.tight_layout()
plt.show()
